# Process Raw Chamber Data

Load raw chamber measurements, map user-specific column names into the FCS schema, preview the data, and run standard FCS processing with optional MCMC. This notebook supports the project JSON folder format and generic CSV files.

## Setup

In [1]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = pathlib.Path.cwd()
if not (NOTEBOOK_DIR / "01_process_raw_data.ipynb").exists():
    NOTEBOOK_DIR = (pathlib.Path.cwd() / "notebooks" / "processing").resolve()

REPO_ROOT = NOTEBOOK_DIR.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from soilgasflux_fcs import Multiprocessor, json_reader

DEFAULT_OUTPUT_DIR = NOTEBOOK_DIR / "output"
DEFAULT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_COLUMNS = [
    "datetime",
    "id",
    "timedelta",
    "k30_co2",
    "si_temperature",
    "si_humidity",
    "bmp_pressure",
]

## Loading And Normalization Helpers

In [2]:
def find_input_files(input_path, pattern="*.csv"):
    path = pathlib.Path(input_path).expanduser()
    if path.is_file():
        return [path]
    if path.is_dir():
        return sorted(path.glob(pattern))
    raise FileNotFoundError(f"Input path does not exist: {path}")


def load_json_folder(folder_path):
    initializer = json_reader.Initializer(folder_path)
    df = initializer.prepare_rawdata()
    return df[CANONICAL_COLUMNS + [c for c in df.columns if c not in CANONICAL_COLUMNS]]


def load_csv_files(input_path, pattern="*.csv", delimiter=","):
    files = find_input_files(input_path, pattern=pattern)
    if not files:
        raise FileNotFoundError(f"No CSV files matched {input_path!r} with pattern {pattern!r}")

    frames = []
    for file in files:
        df = pd.read_csv(file, sep=delimiter)
        df["__source_file"] = file.stem
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def guess_column(columns, candidates):
    normalized = {str(col).lower().strip(): col for col in columns}
    for candidate in candidates:
        key = candidate.lower().strip()
        if key in normalized:
            return normalized[key]
    for col in columns:
        lower = str(col).lower()
        if any(candidate.lower() in lower for candidate in candidates):
            return col
    return ""


def _require_or_fill(raw_df, source_col, output_col, fill_missing_environment, default_value):
    if source_col:
        return raw_df[source_col]
    if fill_missing_environment:
        return default_value
    raise ValueError(
        f"Missing mapping for {output_col}. Select a source column or enable constant-fill mode."
    )


def normalize_csv_dataframe(
    raw_df,
    *,
    co2_col,
    timestamp_col="",
    elapsed_col="",
    id_col="",
    pressure_col="",
    temperature_col="",
    humidity_col="",
    pressure_unit="Pa",
    fill_missing_environment=False,
    default_pressure_pa=101325.0,
    default_temperature_c=20.0,
    default_humidity_percent=70.0,
):
    if not co2_col:
        raise ValueError("CO2 column is required.")
    if not timestamp_col and not elapsed_col:
        raise ValueError("Select either a timestamp column or an elapsed-seconds column.")

    df = pd.DataFrame()
    if id_col:
        df["id"] = raw_df[id_col].astype(str)
    elif "__source_file" in raw_df.columns:
        df["id"] = raw_df["__source_file"].astype(str)
    else:
        df["id"] = "measurement_001"

    df["k30_co2"] = pd.to_numeric(raw_df[co2_col], errors="coerce")

    if timestamp_col:
        df["datetime"] = pd.to_datetime(raw_df[timestamp_col], errors="coerce")
        df["timedelta"] = (
            df.groupby("id")["datetime"]
            .transform(lambda values: (values - values.min()).dt.total_seconds())
            .astype("float")
        )
    else:
        df["timedelta"] = pd.to_numeric(raw_df[elapsed_col], errors="coerce")
        base = pd.Timestamp("2000-01-01")
        offsets = {measurement_id: n for n, measurement_id in enumerate(df["id"].drop_duplicates())}
        df["datetime"] = [
            base + pd.Timedelta(days=offsets[measurement_id]) + pd.Timedelta(seconds=float(seconds))
            if pd.notna(seconds) else pd.NaT
            for measurement_id, seconds in zip(df["id"], df["timedelta"])
        ]

    pressure = _require_or_fill(
        raw_df,
        pressure_col,
        "bmp_pressure",
        fill_missing_environment,
        default_pressure_pa,
    )
    pressure = pd.to_numeric(pressure, errors="coerce")
    if pressure_col and pressure_unit == "kPa":
        pressure = pressure * 1000.0
    df["bmp_pressure"] = pressure

    df["si_temperature"] = pd.to_numeric(
        _require_or_fill(
            raw_df,
            temperature_col,
            "si_temperature",
            fill_missing_environment,
            default_temperature_c,
        ),
        errors="coerce",
    )
    df["si_humidity"] = pd.to_numeric(
        _require_or_fill(
            raw_df,
            humidity_col,
            "si_humidity",
            fill_missing_environment,
            default_humidity_percent,
        ),
        errors="coerce",
    )

    df = df.sort_values(["id", "datetime", "timedelta"]).reset_index(drop=True)
    return df[CANONICAL_COLUMNS]


def validate_fcs_dataframe(df):
    missing = [column for column in CANONICAL_COLUMNS if column not in df.columns]
    null_counts = df[CANONICAL_COLUMNS].isna().sum().to_dict() if not missing else {}
    valid = not missing and all(count == 0 for count in null_counts.values())
    return {
        "valid": valid,
        "missing_columns": missing,
        "null_counts": null_counts,
        "n_measurements": int(df["id"].nunique()) if "id" in df.columns else 0,
        "n_rows": int(len(df)),
    }


def summarize_measurements(df):
    summary = (
        df.groupby("id")
        .agg(
            n_rows=("timedelta", "size"),
            start=("datetime", "min"),
            end=("datetime", "max"),
            duration_s=("timedelta", "max"),
            min_co2=("k30_co2", "min"),
            max_co2=("k30_co2", "max"),
        )
        .reset_index()
    )
    return summary


def plot_measurement(df, measurement_id=None):
    measurement_id = measurement_id or df["id"].iloc[0]
    selected = df[df["id"] == measurement_id]
    fig, ax = plt.subplots(figsize=(7, 3.5), dpi=120)
    ax.plot(selected["timedelta"], selected["k30_co2"], color="#1f77b4", linewidth=1.8)
    ax.scatter(selected["timedelta"], selected["k30_co2"], color="#1f77b4", s=10, alpha=0.4)
    ax.set_xlabel("Elapsed time [s]")
    ax.set_ylabel(r"$CO_2$ [ppm]")
    ax.set_title(f"Measurement: {measurement_id}")
    fig.tight_layout()
    return fig


def run_fcs_processing(
    df,
    *,
    chamber_id,
    output_folder=DEFAULT_OUTPUT_DIR,
    area=314.0,
    volume=6283.0,
    use_mcmc=False,
    n_mc=500,
    sensor_precision=None,
):
    validation = validate_fcs_dataframe(df)
    if not validation["valid"]:
        raise ValueError(f"Dataframe is not ready for FCS processing: {validation}")

    output_folder = pathlib.Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    processor = Multiprocessor()
    metadata = {"area": area, "volume": volume}
    if use_mcmc:
        return processor.run_MC(
            df=df,
            chamber_id=chamber_id,
            output_folder=str(output_folder),
            save_netcdf=True,
            sensor_precision=sensor_precision,
            n_MC=n_mc,
            metadata=metadata,
        )
    return processor.run(
        df=df,
        chamber_id=chamber_id,
        output_folder=str(output_folder),
        metadata=metadata,
    )

## Plain Python Example

Use these calls directly in scripts, or use the widget section below for an interactive workflow.

In [3]:
# JSON folder example
# df = load_json_folder("path/to/json_folder")

# CSV example
# raw = load_csv_files("path/to/csv_folder", pattern="*.csv", delimiter=",")
# df = normalize_csv_dataframe(
#     raw,
#     timestamp_col="timestamp",
#     co2_col="co2",
#     pressure_col="pressure",
#     pressure_unit="Pa",
#     temperature_col="temp",
#     humidity_col="rh",
#     fill_missing_environment=False,
# )
# summarize_measurements(df)

## Interactive Workflow

In [4]:
import ipywidgets as widgets
from IPython.display import clear_output, display

state = {"raw": None, "df": None}
style = {"description_width": "160px"}
wide = widgets.Layout(width="620px")
medium = widgets.Layout(width="430px")

input_mode_widget = widgets.Dropdown(options=["CSV", "JSON folder"], value="CSV", description="Input mode", style=style, layout=medium)
input_path_widget = widgets.Text(value="", description="Input path", placeholder="File or folder path", style=style, layout=wide)
pattern_widget = widgets.Text(value="*.csv", description="CSV pattern", style=style, layout=medium)
delimiter_widget = widgets.Text(value=",", description="CSV delimiter", style=style, layout=medium)

id_col_widget = widgets.Dropdown(options=[""], description="Measurement id", style=style, layout=medium)
timestamp_col_widget = widgets.Dropdown(options=[""], description="Timestamp", style=style, layout=medium)
elapsed_col_widget = widgets.Dropdown(options=[""], description="Elapsed seconds", style=style, layout=medium)
co2_col_widget = widgets.Dropdown(options=[""], description="CO2", style=style, layout=medium)
pressure_col_widget = widgets.Dropdown(options=[""], description="Pressure", style=style, layout=medium)
temperature_col_widget = widgets.Dropdown(options=[""], description="Temperature", style=style, layout=medium)
humidity_col_widget = widgets.Dropdown(options=[""], description="Humidity", style=style, layout=medium)
pressure_unit_widget = widgets.Dropdown(options=["Pa", "kPa"], value="Pa", description="Pressure unit", style=style, layout=medium)
fill_missing_widget = widgets.Checkbox(value=False, description="Allow constant environmental defaults", indent=False, layout=wide)
default_pressure_widget = widgets.FloatText(value=101325.0, description="Default pressure [Pa]", style=style, layout=medium)
default_temperature_widget = widgets.FloatText(value=20.0, description="Default temp [C]", style=style, layout=medium)
default_humidity_widget = widgets.FloatText(value=70.0, description="Default RH [%]", style=style, layout=medium)

measurement_widget = widgets.Dropdown(options=[], description="Preview id", style=style, layout=medium)
chamber_id_widget = widgets.Text(value="analysis", description="Chamber id", style=style, layout=medium)
output_folder_widget = widgets.Text(value=str(DEFAULT_OUTPUT_DIR), description="Output folder", style=style, layout=wide)
area_widget = widgets.FloatText(value=314.0, description="Area [cm2]", style=style, layout=medium)
volume_widget = widgets.FloatText(value=6283.0, description="Volume [cm3]", style=style, layout=medium)
use_mcmc_widget = widgets.Checkbox(value=False, description="Run MCMC", indent=False, layout=medium)
n_mc_widget = widgets.IntText(value=500, description="n_MC", style=style, layout=medium)
sensor_precision_widget = widgets.FloatText(value=np.nan, description="Sensor precision", style=style, layout=medium)

load_button = widgets.Button(description="Load input", button_style="primary")
normalize_button = widgets.Button(description="Normalize + preview", button_style="info")
process_button = widgets.Button(description="Run FCS", button_style="success")
output = widgets.Output()


def _set_column_options(columns):
    options = [""] + list(columns)
    widgets_to_update = [
        id_col_widget,
        timestamp_col_widget,
        elapsed_col_widget,
        co2_col_widget,
        pressure_col_widget,
        temperature_col_widget,
        humidity_col_widget,
    ]
    for widget in widgets_to_update:
        widget.options = options

    id_col_widget.value = guess_column(columns, ["id", "measurement", "measurement_id", "chamber"])
    timestamp_col_widget.value = guess_column(columns, ["timestamp", "datetime", "datetime_utc", "time"])
    elapsed_col_widget.value = guess_column(columns, ["timedelta", "elapsed", "seconds", "time_s"])
    co2_col_widget.value = guess_column(columns, ["co2", "k30_co2", "co2_ppm"])
    pressure_col_widget.value = guess_column(columns, ["pressure", "bmp_pressure", "chamber_p"])
    temperature_col_widget.value = guess_column(columns, ["temperature", "temp", "si_temperature", "chamber_t"])
    humidity_col_widget.value = guess_column(columns, ["humidity", "rh", "si_humidity"])


def on_load_clicked(_):
    with output:
        clear_output(wait=True)
        if input_mode_widget.value == "JSON folder":
            df = load_json_folder(input_path_widget.value)
            state["raw"] = df
            state["df"] = df
            measurement_widget.options = sorted(df["id"].astype(str).unique())
            display(df.head())
            display(summarize_measurements(df))
            print("Loaded JSON folder and normalized to FCS schema.")
            return

        raw = load_csv_files(
            input_path_widget.value,
            pattern=pattern_widget.value,
            delimiter=delimiter_widget.value,
        )
        state["raw"] = raw
        state["df"] = None
        _set_column_options(raw.columns)
        display(raw.head())
        print(f"Detected {len(raw.columns)} columns and {len(raw)} rows.")


def on_normalize_clicked(_):
    with output:
        clear_output(wait=True)
        if input_mode_widget.value == "JSON folder":
            if state["df"] is None:
                on_load_clicked(None)
            df = state["df"]
        else:
            if state["raw"] is None:
                raise ValueError("Load CSV input before normalizing.")
            df = normalize_csv_dataframe(
                state["raw"],
                id_col=id_col_widget.value,
                timestamp_col=timestamp_col_widget.value,
                elapsed_col=elapsed_col_widget.value,
                co2_col=co2_col_widget.value,
                pressure_col=pressure_col_widget.value,
                temperature_col=temperature_col_widget.value,
                humidity_col=humidity_col_widget.value,
                pressure_unit=pressure_unit_widget.value,
                fill_missing_environment=fill_missing_widget.value,
                default_pressure_pa=default_pressure_widget.value,
                default_temperature_c=default_temperature_widget.value,
                default_humidity_percent=default_humidity_widget.value,
            )
            state["df"] = df

        validation = validate_fcs_dataframe(df)
        measurement_widget.options = sorted(df["id"].astype(str).unique())
        display(df.head())
        display(summarize_measurements(df))
        display(validation)
        if validation["valid"] and measurement_widget.options:
            display(plot_measurement(df, measurement_widget.value))


def on_process_clicked(_):
    with output:
        clear_output(wait=True)
        if state["df"] is None:
            raise ValueError("Normalize data before running FCS.")
        sensor_precision = None if np.isnan(sensor_precision_widget.value) else sensor_precision_widget.value
        result = run_fcs_processing(
            state["df"],
            chamber_id=chamber_id_widget.value,
            output_folder=output_folder_widget.value,
            area=area_widget.value,
            volume=volume_widget.value,
            use_mcmc=use_mcmc_widget.value,
            n_mc=n_mc_widget.value,
            sensor_precision=sensor_precision,
        )
        print("FCS processing complete.")
        print(f"Output folder: {output_folder_widget.value}")
        display(result)


load_button.on_click(on_load_clicked)
normalize_button.on_click(on_normalize_clicked)
process_button.on_click(on_process_clicked)

input_controls = widgets.VBox([
    input_mode_widget,
    input_path_widget,
    widgets.HBox([pattern_widget, delimiter_widget]),
    widgets.HBox([load_button, normalize_button]),
])

mapping_controls = widgets.VBox([
    widgets.HBox([id_col_widget, timestamp_col_widget, elapsed_col_widget]),
    widgets.HBox([co2_col_widget, pressure_col_widget, pressure_unit_widget]),
    widgets.HBox([temperature_col_widget, humidity_col_widget]),
    fill_missing_widget,
    widgets.HBox([default_pressure_widget, default_temperature_widget, default_humidity_widget]),
])

processing_controls = widgets.VBox([
    widgets.HBox([measurement_widget, chamber_id_widget]),
    output_folder_widget,
    widgets.HBox([area_widget, volume_widget]),
    widgets.HBox([use_mcmc_widget, n_mc_widget, sensor_precision_widget]),
    process_button,
])

display(widgets.Tab(children=[input_controls, mapping_controls, processing_controls]), output)

Output()